# Phase 5: 멀티 포지션 파이프라인

ClientRequest 하나로 포지션별 쿼리 생성 → 하이브리드 검색 → 중복 해소까지 (STEP 1~3)

In [6]:
from _bootstrap import setup_project_path

setup_project_path()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


WindowsPath('C:/Users/mk.jang/Desktop/TLC/08_TSM/retrieval-lab')

In [7]:
from embedding_retrieval.config import RetrievalConfig
from embedding_retrieval.factory import create_embeddings
from embedding_retrieval.stores.dual_upstash import DualUpstashStore
from embedding_retrieval.search.bm25 import BM25Index
from embedding_retrieval.search.hybrid import HybridSearcher
from embedding_retrieval.pipeline.recommendation import RecommendationPipeline
from embedding_retrieval.scenarios.sample_data import SAMPLE_ENGINEER_PROFILES
from embedding_retrieval.types import ClientRequest, PositionRequest

config = RetrievalConfig.from_env()
embeddings = create_embeddings(config)
url = config.vector_store_kwargs["url"]
token = config.vector_store_kwargs["token"]

# 인프라 셋업
store = DualUpstashStore(embeddings=embeddings, url=url, token=token)

cap_bm25 = BM25Index()
cap_bm25.fit([(p.engineer_id, p.capability_text) for p in SAMPLE_ENGINEER_PROFILES])
exp_bm25 = BM25Index()
exp_bm25.fit([(p.engineer_id, p.experience_text) for p in SAMPLE_ENGINEER_PROFILES])

searcher = HybridSearcher(store=store, cap_bm25=cap_bm25, exp_bm25=exp_bm25, profiles=SAMPLE_ENGINEER_PROFILES)
pipeline = RecommendationPipeline(searcher=searcher)
print("파이프라인 준비 완료")

파이프라인 준비 완료


In [8]:
# 쿼리 빌더 확인
from embedding_retrieval.pipeline.query_builder import build_queries

request = ClientRequest(
    project_name="현대차 ERP 시스템 개발",
    description="제조업 ERP 시스템 재구축 프로젝트",
    weights={"capability": 0.6, "experience": 0.4},
    positions=[
        PositionRequest(position="PL", engineer_role="DEVELOPER", grades=["SENIOR", "INTERMEDIATE"], skills=["Java", "Spring"], engineer_cnt=1, etc="팀장 역할 필수"),
        PositionRequest(position="백엔드 개발자", engineer_role="DEVELOPER", grades=["SENIOR", "INTERMEDIATE"], skills=["Java", "Spring"], engineer_cnt=3, etc="현대차 프로젝트 경험 우대"),
        PositionRequest(position="프론트엔드 개발자", engineer_role="DEVELOPER", grades=["SENIOR", "INTERMEDIATE"], skills=["React"], engineer_cnt=2, etc="차트.js 경험 우대"),
    ],
)

queries = build_queries(request)
for q in queries:
    print(f"--- {q['position']} ---")
    print(f"  capability: {q['capability_query']}")
    print(f"  experience: {q['experience_query']}")
    print()

--- PL ---
  capability: Java / Spring
  experience: [프로젝트] 현대차 ERP 시스템 개발: 제조업 ERP 시스템 재구축 프로젝트
[포지션] PL
[요건] 팀장 역할 필수

--- 백엔드 개발자 ---
  capability: Java / Spring
  experience: [프로젝트] 현대차 ERP 시스템 개발: 제조업 ERP 시스템 재구축 프로젝트
[포지션] 백엔드 개발자
[요건] 현대차 프로젝트 경험 우대

--- 프론트엔드 개발자 ---
  capability: React
  experience: [프로젝트] 현대차 ERP 시스템 개발: 제조업 ERP 시스템 재구축 프로젝트
[포지션] 프론트엔드 개발자
[요건] 차트.js 경험 우대



In [9]:
# 전체 파이프라인 실행
response = pipeline.recommend(request)

print(f"request_id: {response.request_id}")
print(f"포지션 수: {len(response.positions)}")

for pr in response.positions:
    print(f"\n{'='*60}")
    print(f"[{pr.position}] {len(pr.candidates)}명 추천")
    print(f"{'='*60}")
    for c in pr.candidates:
        s = c.score_breakdown
        print(f"  #{c.rank} {c.engineer_id:8s} | final: {s.final_score:.5f} | grade: {c.profile.grade}")
        print(f"       cap_d: {s.capability_dense:.4f}  cap_b: {s.capability_bm25:.4f}  exp_d: {s.experience_dense:.4f}  exp_b: {s.experience_bm25:.4f}")

request_id: df0be5435b38
포지션 수: 3

[PL] 1명 추천
  #1 eng-001  | final: 0.01634 | grade: SENIOR
       cap_d: 0.9091  cap_b: 3.0422  exp_d: 0.8974  exp_b: 9.9742

[백엔드 개발자] 3명 추천
  #1 eng-002  | final: 0.01631 | grade: INTERMEDIATE
       cap_d: 0.9576  cap_b: 3.1960  exp_d: 0.9122  exp_b: 7.7875
  #2 eng-005  | final: 0.01028 | grade: INTERMEDIATE
       cap_d: 0.8876  cap_b: 0.0000  exp_d: 0.8855  exp_b: 3.3177
  #3 eng-003  | final: 0.01019 | grade: SENIOR
       cap_d: 0.8629  cap_b: 0.0000  exp_d: 0.8860  exp_b: 0.2383

[프론트엔드 개발자] 0명 추천


In [10]:
# 중복 해소 확인: 같은 엔지니어가 여러 포지션에 나오지 않는지
all_ids = []
for pr in response.positions:
    ids = [c.engineer_id for c in pr.candidates]
    print(f"{pr.position}: {ids}")
    all_ids.extend(ids)

duplicates = [eid for eid in all_ids if all_ids.count(eid) > 1]
if duplicates:
    print(f"\n⚠️ 중복 발견: {set(duplicates)}")
else:
    print(f"\n✅ 중복 없음 — {len(all_ids)}명 모두 고유")

PL: ['eng-001']
백엔드 개발자: ['eng-002', 'eng-005', 'eng-003']
프론트엔드 개발자: []

✅ 중복 없음 — 4명 모두 고유
